#Comparación de modelos de clasificación para datos de contaminante aereo NOx y datos de urgencias respiratorios en la quinta región

## 1. Imports

In [0]:
from functools import reduce
import re
import unicodedata

from pyspark.sql import functions as F
from pyspark.sql import Window

## 2. Carga de datos

In [0]:
CATALOG = "upla"
SCHEMA = "mcdma_analisis_datos_ma"
VOLUME = "aire_salud_volume"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
SINCA_DIR = f"{BASE_PATH}/raw/sinca"
URGENCIAS_DIR = f"{BASE_PATH}/raw/urgencias"

BRONZE_SINCA_TABLE = f"{CATALOG}.{SCHEMA}.bronze_sinca_nox_diario"
BRONZE_URGENCIAS_TABLE = f"{CATALOG}.{SCHEMA}.bronze_urgencias"
SILVER_SINCA_TABLE = f"{CATALOG}.{SCHEMA}.silver_sinca_nox_semanal"
SILVER_URGENCIAS_TABLE = f"{CATALOG}.{SCHEMA}.silver_urgencias_resp_semanal"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_aire_salud_semanal"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("BASE_PATH:", BASE_PATH)
print("SINCA_DIR:", SINCA_DIR)
print("URGENCIAS_DIR:", URGENCIAS_DIR)

BASE_PATH: /Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume
SINCA_DIR: /Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca
URGENCIAS_DIR: /Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias


In [0]:
display(dbutils.fs.ls(SINCA_DIR))
display(dbutils.fs.ls(URGENCIAS_DIR))

path,name,size,modificationTime
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_centro_quintero_2018_2025.csv,v_nox_diario_centro_quintero_2018_2025.csv,66851,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_colmo_2018_2025.csv,v_nox_diario_colmo_2018_2025.csv,65893,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_concon_2018_2025.csv,v_nox_diario_concon_2018_2025.csv,66632,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_cuerpo_de_bomberos_2018_2025.csv,v_nox_diario_cuerpo_de_bomberos_2018_2025.csv,65744,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_la_greda_2018_2025.csv,v_nox_diario_la_greda_2018_2025.csv,66696,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_la_palma_2018_2025.csv,v_nox_diario_la_palma_2018_2025.csv,65295,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_loncura_2018_2025.csv,v_nox_diario_loncura_2018_2025.csv,63205,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_los_andes_2018_2025.csv,v_nox_diario_los_andes_2018_2025.csv,62665,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_los_maitenes_2018_2025.csv,v_nox_diario_los_maitenes_2018_2025.csv,66687,1781149870000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_puchuncavi_2018_2025.csv,v_nox_diario_puchuncavi_2018_2025.csv,66693,1781149870000


path,name,size,modificationTime
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2020.csv,AtencionesUrgencia2020.csv,796227186,1781149926000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2021.csv,AtencionesUrgencia2021.csv,1130831814,1781149926000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2022.csv,AtencionesUrgencia2022.csv,1144770884,1781149926000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2023.csv,AtencionesUrgencia2023.csv,1598833998,1781149926000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2024.csv,AtencionesUrgencia2024.csv,1608911448,1781149926000
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2025.csv,AtencionesUrgencia2025.csv,1784414515,1781149926000


In [0]:
def clean_col_name(c):
    c = c.strip()
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode("ascii")
    c = re.sub(r"[^0-9A-Za-z]+", "_", c)
    c = c.strip("_").lower()
    return c


def make_unique_columns(cols):
    clean_cols = []
    seen = {}

    for i, c in enumerate(cols):
        name = clean_col_name(c)

        if not name:
            name = f"empty_col_{i}"

        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0

        clean_cols.append(name)

    return clean_cols


def read_csv_folder(folder_path, encoding="UTF-8"):
    files = [
        f.path for f in dbutils.fs.ls(folder_path)
        if f.name.lower().endswith(".csv")
    ]

    print("Archivos encontrados:", len(files))

    dfs = []

    for path in files:
        print("Leyendo:", path)

        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("sep", ";")
            .option("encoding", encoding)
            .option("inferSchema", "false")
            .load(path)
        )

        df = df.toDF(*make_unique_columns(df.columns))
        df = df.withColumn("source_file", F.lit(path.split("/")[-1]))
        dfs.append(df)

    all_cols = sorted(set().union(*[set(df.columns) for df in dfs]))

    dfs_aligned = [
        df.select([
            F.col(c) if c in df.columns else F.lit(None).alias(c)
            for c in all_cols
        ])
        for df in dfs
    ]

    return reduce(lambda a, b: a.unionByName(b), dfs_aligned)


def to_double_comma(col_name):
    return (
        F.regexp_replace(
            F.when(F.trim(F.col(col_name).cast("string")) == "", None)
             .otherwise(F.trim(F.col(col_name).cast("string"))),
            ",",
            "."
        )
        .cast("double")
    )


def to_long_safe(col_name):
    return (
        F.regexp_replace(
            F.when(F.trim(F.col(col_name).cast("string")) == "", None)
             .otherwise(F.trim(F.col(col_name).cast("string"))),
            ",",
            "."
        )
        .cast("double")
        .cast("long")
    )

In [0]:
sinca_raw = read_csv_folder(SINCA_DIR, encoding="UTF-8")

print(sinca_raw.columns)
display(sinca_raw.limit(10))

Archivos encontrados: 15
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_centro_quintero_2018_2025.csv
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_colmo_2018_2025.csv
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_concon_2018_2025.csv
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_cuerpo_de_bomberos_2018_2025.csv
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_la_greda_2018_2025.csv
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_la_palma_2018_2025.csv
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_loncura_2018_2025.csv
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/sinca/v_nox_diario_los_andes_2018_2025.csv
Leyendo: dbfs:/Volumes/upla/mcdma_a

c5,fecha_yymmdd,hora_hhmm,registros_no_validados,registros_preliminares,registros_validados,source_file
null,180101,0000,"9,03348",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180102,0000,"12,2869",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180103,0000,"14,4776",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180104,0000,"16,9356",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180105,0000,"4,82915",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180106,0000,"11,1457",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180107,0000,"7,0108",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180108,0000,"10,5179",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180109,0000,"10,0118",null,null,v_nox_diario_centro_quintero_2018_2025.csv
null,180110,0000,"7,57923",null,null,v_nox_diario_centro_quintero_2018_2025.csv


In [0]:
# Esta configuracion es porque los archivos pesan hasta 1.7 GB y se deben cargar de a poco para revisar y despues filtrar
spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)

print("Configuración lista para archivos grandes")

Configuración lista para archivos grandes


In [0]:
def read_urgencias_file(path, encoding="UTF-8"):
    print("Leyendo:", path)

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", encoding)
        .option("inferSchema", "false")
        .load(path)
    )

    df = df.toDF(*make_unique_columns(df.columns))

    required_cols = [
        "idestablecimiento",
        "nestablecimiento",
        "idcausa",
        "glosacausa",
        "total",
        "menores_1",
        "de_1_a_4",
        "de_5_a_14",
        "de_15_a_64",
        "de_65_y_mas",
        "fecha",
        "semana",
        "glosatipoestablecimiento",
        "glosatipoatencion",
        "glosatipocampana",
        "codigoregion",
        "nombreregion",
        "codigodependencia",
        "nombredependencia",
        "codigocomuna",
        "nombrecomuna"
    ]

    df = df.select([
        F.col(c) if c in df.columns else F.lit(None).alias(c)
        for c in required_cols
    ])

    df = (
        df
        .withColumn("source_file", F.lit(path.split("/")[-1]))
        .withColumn("fecha", F.to_date(F.col("fecha"), "dd/MM/yyyy"))
        .withColumn("anio", F.year("fecha"))
        .withColumn("semana_minsal", F.col("semana").cast("int"))
        .withColumn("idcausa_int", F.col("idcausa").cast("int"))
        .withColumn("codigoregion_int", F.col("codigoregion").cast("int"))
        .withColumn("codigocomuna_int", F.col("codigocomuna").cast("int"))
        .withColumn("total_num", to_long_safe("total"))
        .withColumn("menores_1_num", to_long_safe("menores_1"))
        .withColumn("de_1_a_4_num", to_long_safe("de_1_a_4"))
        .withColumn("de_5_a_14_num", to_long_safe("de_5_a_14"))
        .withColumn("de_15_a_64_num", to_long_safe("de_15_a_64"))
        .withColumn("de_65_y_mas_num", to_long_safe("de_65_y_mas"))
        .filter(F.col("fecha").isNotNull())
        .filter((F.col("anio") >= 2020) & (F.col("anio") <= 2025))
        .filter(
            (F.col("codigoregion_int") == 5) |
            (F.upper(F.col("nombreregion")).contains("VALPAR"))
        )
    )

    return df

def select_first_existing(df, candidates, alias_name):
    """
    Selecciona la primera columna existente entre varios nombres posibles.
    Si ninguna existe, crea la columna como null.
    """
    for c in candidates:
        if c in df.columns:
            return F.col(c).alias(alias_name)
    return F.lit(None).alias(alias_name)


def read_urgencias_file_v2(path, encoding="ISO-8859-1"):
    print("Leyendo:", path.split("/")[-1])

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", encoding)
        .option("inferSchema", "false")
        .load(path)
    )

    df = df.toDF(*make_unique_columns(df.columns))

    df = df.select(
        select_first_existing(df, ["idestablecimiento", "id_establecimiento"], "idestablecimiento"),
        select_first_existing(df, ["nestablecimiento", "n_establecimiento", "nombre_establecimiento"], "nestablecimiento"),
        select_first_existing(df, ["idcausa", "id_causa"], "idcausa"),
        select_first_existing(df, ["glosacausa", "glosa_causa"], "glosacausa"),
        select_first_existing(df, ["total"], "total"),
        select_first_existing(df, ["menores_1", "menor_a_1", "menor_1"], "menores_1"),
        select_first_existing(df, ["de_1_a_4", "column7"], "de_1_a_4"),
        select_first_existing(df, ["de_5_a_14", "_14"], "de_5_a_14"),
        select_first_existing(df, ["de_15_a_64", "_5_64"], "de_15_a_64"),
        select_first_existing(df, ["de_65_y_mas", "_5_mas", "de_65_y_mas_"], "de_65_y_mas"),
        select_first_existing(df, ["fecha"], "fecha"),
        select_first_existing(df, ["semana"], "semana"),
        select_first_existing(df, ["glosatipoestablecimiento"], "glosatipoestablecimiento"),
        select_first_existing(df, ["glosatipoatencion"], "glosatipoatencion"),
        select_first_existing(df, ["glosatipocampana"], "glosatipocampana"),
        select_first_existing(df, ["codigoregion", "codigo_region"], "codigoregion"),
        select_first_existing(df, ["nombreregion", "nombre_region"], "nombreregion"),
        select_first_existing(df, ["codigodependencia", "codigo_dependencia"], "codigodependencia"),
        select_first_existing(df, ["nombredependencia", "nombre_dependencia"], "nombredependencia"),
        select_first_existing(df, ["codigocomuna", "codigo_comuna"], "codigocomuna"),
        select_first_existing(df, ["nombrecomuna", "nombre_comuna"], "nombrecomuna")
    )

    df = (
        df
        .withColumn("source_file", F.lit(path.split("/")[-1]))
        .withColumn("fecha", F.to_date(F.col("fecha"), "dd/MM/yyyy"))
        .withColumn("anio", F.year("fecha"))
        .withColumn("semana_minsal", F.col("semana").cast("int"))
        .withColumn("idcausa_int", F.col("idcausa").cast("int"))
        .withColumn("codigoregion_int", F.col("codigoregion").cast("int"))
        .withColumn("codigocomuna_int", F.col("codigocomuna").cast("int"))
        .withColumn("total_num", to_long_safe("total"))
        .withColumn("menores_1_num", to_long_safe("menores_1"))
        .withColumn("de_1_a_4_num", to_long_safe("de_1_a_4"))
        .withColumn("de_5_a_14_num", to_long_safe("de_5_a_14"))
        .withColumn("de_15_a_64_num", to_long_safe("de_15_a_64"))
        .withColumn("de_65_y_mas_num", to_long_safe("de_65_y_mas"))
        .filter(F.col("fecha").isNotNull())
        .filter((F.col("anio") >= 2020) & (F.col("anio") <= 2025))
    )

    return df

In [0]:
urg_files = [
    f.path for f in dbutils.fs.ls(URGENCIAS_DIR)
    if f.name.lower().endswith(".csv")
]

print("Archivos de urgencias:", len(urg_files))
for f in urg_files:
    print(f)

urg_dfs = [read_urgencias_file(path, encoding="UTF-8") for path in urg_files]

urg_valpo = reduce(lambda a, b: a.unionByName(b), urg_dfs)

print("DataFrame urg_valpo creado")

Archivos de urgencias: 6
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2020.csv
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2021.csv
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2022.csv
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2023.csv
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2024.csv
dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2025.csv
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2020.csv
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2021.csv
Leyendo: dbfs:/Volumes/upla/mcdma_analisis_datos_ma/aire_salud_volume/raw/urgencias/AtencionesUrgencia2022.csv
Leyendo: dbfs:/Volumes/upla/mc

## 3. Preprocesamiento

In [0]:
sinca_clean = (
    sinca_raw
    .withColumn(
        "fecha_str",
        F.lpad(F.col("fecha_yymmdd").cast("string"), 6, "0")
    )
    .withColumn(
        "fecha",
        F.to_date(
            F.concat(
                F.lit("20"),
                F.substring("fecha_str", 1, 2),
                F.substring("fecha_str", 3, 2),
                F.substring("fecha_str", 5, 2)
            ),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "estacion",
        F.regexp_replace(
            F.regexp_replace(F.col("source_file"), r"^v_nox_diario_", ""),
            r"_2018_2025\.csv$",
            ""
        )
    )
    .withColumn("valor_validado", to_double_comma("registros_validados"))
    .withColumn("valor_preliminar", to_double_comma("registros_preliminares"))
    .withColumn("valor_no_validado", to_double_comma("registros_no_validados"))
    .withColumn(
        "valor_nox",
        F.coalesce("valor_validado", "valor_preliminar", "valor_no_validado")
    )
    .withColumn(
        "tipo_registro",
        F.when(F.col("valor_validado").isNotNull(), F.lit("validado"))
         .when(F.col("valor_preliminar").isNotNull(), F.lit("preliminar"))
         .when(F.col("valor_no_validado").isNotNull(), F.lit("no_validado"))
         .otherwise(F.lit("sin_dato"))
    )
    .filter(F.col("fecha").isNotNull())
    .filter(F.col("valor_nox").isNotNull())
    .filter((F.col("fecha") >= F.lit("2020-01-01")) & (F.col("fecha") <= F.lit("2025-12-31")))
    .select(
        "fecha",
        "estacion",
        F.lit("NOx").alias("contaminante"),
        "valor_nox",
        "tipo_registro",
        "source_file"
    )
)

display(sinca_clean.limit(20))

fecha,estacion,contaminante,valor_nox,tipo_registro,source_file
2020-01-01,centro_quintero,NOx,6.19751,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-02,centro_quintero,NOx,8.90251,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-03,centro_quintero,NOx,6.38358,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-04,centro_quintero,NOx,6.74619,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-05,centro_quintero,NOx,6.12192,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-06,centro_quintero,NOx,6.59077,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-07,centro_quintero,NOx,9.35983,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-08,centro_quintero,NOx,6.2711,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-09,centro_quintero,NOx,12.3439,no_validado,v_nox_diario_centro_quintero_2018_2025.csv
2020-01-10,centro_quintero,NOx,10.0713,no_validado,v_nox_diario_centro_quintero_2018_2025.csv


In [0]:
(
    sinca_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_SINCA_TABLE)
)

print("Tabla creada:", BRONZE_SINCA_TABLE)

Tabla creada: upla.mcdma_analisis_datos_ma.bronze_sinca_nox_diario


In [0]:
(
    urg_valpo
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_URGENCIAS_TABLE)
)

print("Tabla creada:", BRONZE_URGENCIAS_TABLE)

Tabla creada: upla.mcdma_analisis_datos_ma.bronze_urgencias


In [0]:
urg_valpo = spark.table(BRONZE_URGENCIAS_TABLE)

In [0]:
urg_resp = urg_valpo.filter(F.col("idcausa_int") == 2)

In [0]:
# Los años 2020 a 2022 no traen el campo comuna, por lo que hay que obtenerlo mediante el codigo de comuna

for path in urg_files:
    df_tmp = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")
        .option("inferSchema", "false")
        .load(path)
    )

    print("\nArchivo:", path.split("/")[-1])
    print(make_unique_columns(df_tmp.columns))


Archivo: AtencionesUrgencia2020.csv
['idestablecimiento', 'nestablecimiento', 'idcausa', 'glosacausa', 'total', 'menores_1', 'de_1_a_4', 'de_5_a_14', 'de_15_a_64', 'de_65_y_mas', 'fecha', 'semana', 'glosatipoestablecimiento', 'glosatipoatencion', 'glosatipocampana']

Archivo: AtencionesUrgencia2021.csv
['idestablecimiento', 'nestablecimiento', 'idcausa', 'glosacausa', 'total', 'menores_1', 'de_1_a_4', 'de_5_a_14', 'de_15_a_64', 'de_65_y_mas', 'fecha', 'semana', 'glosatipoestablecimiento', 'glosatipoatencion', 'glosatipocampana']

Archivo: AtencionesUrgencia2022.csv
['idestablecimiento', 'nestablecimiento', 'idcausa', 'glosacausa', 'total', 'menores_1', 'de_1_a_4', 'de_5_a_14', 'de_15_a_64', 'de_65_y_mas', 'fecha', 'semana', 'glosatipoestablecimiento', 'glosatipoatencion', 'glosatipocampana']

Archivo: AtencionesUrgencia2023.csv
['idestablecimiento', 'nestablecimiento', 'idcausa', 'glosacausa', 'total', 'menores_1', 'de_1_a_4', 'de_5_a_14', 'de_15_a_64', 'de_65_y_mas', 'fecha', 'semana

In [0]:
urg_dfs_v2 = [read_urgencias_file_v2(path, encoding="ISO-8859-1") for path in urg_files]

urg_all = reduce(lambda a, b: a.unionByName(b), urg_dfs_v2)

print("DataFrame urg_all creado")

Leyendo: AtencionesUrgencia2020.csv
Leyendo: AtencionesUrgencia2021.csv
Leyendo: AtencionesUrgencia2022.csv
Leyendo: AtencionesUrgencia2023.csv
Leyendo: AtencionesUrgencia2024.csv
Leyendo: AtencionesUrgencia2025.csv
DataFrame urg_all creado


In [0]:
# Se construye una tabla puente para unificar los datos que tienen la comuna como texto y otros como numero

establecimientos_base = (
    urg_all
    .filter(F.col("codigoregion_int").isNotNull())
    .filter(F.col("codigocomuna_int").isNotNull())
    .withColumn("idestablecimiento_norm", F.trim(F.col("idestablecimiento")))
    .groupBy(
        "idestablecimiento_norm",
        "codigoregion_int",
        "nombreregion",
        "codigocomuna_int",
        "nombrecomuna"
    )
    .agg(F.count("*").alias("n_apariciones"))
)

w_est = Window.partitionBy("idestablecimiento_norm").orderBy(F.desc("n_apariciones"))

establecimientos_map = (
    establecimientos_base
    .withColumn("rn", F.row_number().over(w_est))
    .filter(F.col("rn") == 1)
    .select(
        F.col("idestablecimiento_norm"),
        F.col("codigoregion_int").alias("map_codigoregion"),
        F.col("nombreregion").alias("map_nombreregion"),
        F.col("codigocomuna_int").alias("map_codigocomuna"),
        F.col("nombrecomuna").alias("map_nombrecomuna")
    )
)

display(establecimientos_map.limit(20))

idestablecimiento_norm,map_codigoregion,map_nombreregion,map_codigocomuna,map_nombrecomuna
01-100,15,De Arica y Parinacota,15101,Arica
01-802,15,De Arica y Parinacota,15101,Arica
01-901,15,De Arica y Parinacota,15101,Arica
01-912,15,De Arica y Parinacota,15201,Putre
02-100,1,De Tarapacá,1101,Iquique
02-800,1,De Tarapacá,1101,Iquique
02-801,1,De Tarapacá,1101,Iquique
02-802,1,De Tarapacá,1101,Iquique
02-803,1,De Tarapacá,1401,Pozo Almonte
02-806,1,De Tarapacá,1101,Iquique


In [0]:
urg_all_norm = (
    urg_all
    .withColumn("idestablecimiento_norm", F.trim(F.col("idestablecimiento")))
)

urg_all_filled = (
    urg_all_norm.alias("u")
    .join(
        establecimientos_map.alias("m"),
        on="idestablecimiento_norm",
        how="left"
    )
    .withColumn(
        "codigoregion_final",
        F.coalesce(F.col("u.codigoregion_int"), F.col("m.map_codigoregion"))
    )
    .withColumn(
        "nombreregion_final",
        F.coalesce(F.col("u.nombreregion"), F.col("m.map_nombreregion"))
    )
    .withColumn(
        "codigocomuna_final",
        F.coalesce(F.col("u.codigocomuna_int"), F.col("m.map_codigocomuna"))
    )
    .withColumn(
        "nombrecomuna_final",
        F.coalesce(F.col("u.nombrecomuna"), F.col("m.map_nombrecomuna"))
    )
)

In [0]:
(
    urg_valpo
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_URGENCIAS_TABLE)
)

print("Tabla corregida:", BRONZE_URGENCIAS_TABLE)

## 4. Entrenamiento

No hay celdas de entrenamiento en la version actual del notebook.

## 5. Evaluacion

In [0]:
display(
    sinca_clean
    .groupBy("tipo_registro")
    .count()
)

display(
    sinca_clean
    .groupBy("estacion")
    .agg(
        F.count("*").alias("n_registros"),
        F.min("fecha").alias("fecha_min"),
        F.max("fecha").alias("fecha_max"),
        F.avg("valor_nox").alias("nox_promedio"),
        F.max("valor_nox").alias("nox_maximo")
    )
    .orderBy("estacion")
)

tipo_registro,count
no_validado,29669
preliminar,661
validado,1850


estacion,n_registros,fecha_min,fecha_max,nox_promedio,nox_maximo
centro_quintero,2184,2020-01-01,2025-12-30,14.377350123626387,54.7701
colmo,2162,2020-01-01,2025-12-30,14.63443065217391,84.2017
concon,2153,2020-01-01,2025-12-30,17.51840104505341,96.2332
cuerpo_de_bomberos,2074,2020-01-01,2025-12-30,19.92206397299907,71.4554
la_greda,2161,2020-01-01,2025-12-30,15.87636989356782,61.5365
la_palma,2070,2020-01-01,2025-12-30,9.44697606280194,61.3524
loncura,2139,2020-01-01,2025-12-30,8.841431425899941,40.925
los_andes,2167,2020-01-01,2025-12-30,14.111477798800212,67.9076
los_maitenes,2153,2020-01-01,2025-12-30,9.791911560613109,34.5511
puchuncavi,2159,2020-01-01,2025-12-30,16.473149620194548,49.4931


In [0]:
display(
    urg_valpo
    .groupBy("anio", "nombreregion")
    .agg(
        F.sum("total_num").alias("total_atenciones"),
        F.count("*").alias("filas")
    )
    .orderBy("anio")
)

anio,nombreregion,total_atenciones,filas
2023,De Valpara�so,6822155,911680
2024,De Valpara�so,6870795,904760
2025,De Valpara�so,6759210,957673


In [0]:
resp_candidates = (
    urg_valpo
    .filter(
        F.upper(F.col("glosacausa")).contains("RESPIRATOR") |
        F.upper(F.col("glosacausa")).contains("BRONQUIT") |
        F.upper(F.col("glosacausa")).contains("NEUMON") |
        F.upper(F.col("glosacausa")).contains("INFLUENZA") |
        F.upper(F.col("glosacausa")).contains("IRA")
    )
    .groupBy("anio", "idcausa_int", "glosacausa")
    .agg(
        F.sum("total_num").alias("total"),
        F.count("*").alias("filas")
    )
    .orderBy("anio", "idcausa_int")
)

display(resp_candidates)

anio,idcausa_int,glosacausa,total,filas
2023,2,TOTAL CAUSAS SISTEMA RESPIRATORIO,524683,22792
2023,3,Bronquitis/bronquiolitis aguda (J20-J21),74935,22792
2023,4,Influenza (J09-J11),7464,22792
2023,5,Neumon�a (J12-J18),21185,22792
2023,6,"Otra causa respiratoria (J22, J30-J39, J47, J60-J98)",33859,22792
2023,7,CAUSAS SISTEMA RESPIRATORIO,9370,22792
2023,10,IRA Alta (J00-J06),362241,22792
2024,2,TOTAL CAUSAS SISTEMA RESPIRATORIO,507093,22619
2024,3,Bronquitis/bronquiolitis aguda (J20-J21),79889,22619
2024,4,Influenza (J09-J11),17178,22619


In [0]:
display(
    urg_all
    .groupBy("anio", "source_file")
    .agg(
        F.count("*").alias("filas"),
        F.sum(F.when(F.col("codigoregion_int").isNotNull(), 1).otherwise(0)).alias("filas_con_region"),
        F.sum(F.when(F.col("codigocomuna_int").isNotNull(), 1).otherwise(0)).alias("filas_con_comuna")
    )
    .orderBy("anio", "source_file")
)

anio,source_file,filas,filas_con_region,filas_con_comuna
2020,AtencionesUrgencia2020.csv,6446646,0,0
2021,AtencionesUrgencia2021.csv,8816240,0,0
2022,AtencionesUrgencia2022.csv,8926307,0,0
2023,AtencionesUrgencia2023.csv,8899080,8899080,8899080
2024,AtencionesUrgencia2024.csv,8973229,8965509,8965509
2025,AtencionesUrgencia2025.csv,9142479,9142239,9142239


In [0]:
print("Establecimientos mapeados:", establecimientos_map.count())

display(
    establecimientos_map
    .groupBy("map_codigoregion", "map_nombreregion")
    .count()
    .orderBy("map_codigoregion")
)

Establecimientos mapeados: 667


map_codigoregion,map_nombreregion,count
1,De Tarapacá,21
2,De Antofagasta,22
3,De Atacama,15
4,De Coquimbo,34
5,De Valparaíso,80
6,Del Libertador B. O'Higgins,45
7,Del Maule,60
8,Del Bíobío,70
9,De La Araucanía,42
10,De Los Lagos,45


In [0]:
display(
    urg_all_filled
    .groupBy("anio")
    .agg(
        F.count("*").alias("filas"),
        F.sum(F.when(F.col("codigoregion_final").isNotNull(), 1).otherwise(0)).alias("filas_con_region_final"),
        F.sum(F.when(F.col("codigocomuna_final").isNotNull(), 1).otherwise(0)).alias("filas_con_comuna_final"),
        F.countDistinct("idestablecimiento_norm").alias("n_establecimientos")
    )
    .orderBy("anio")
)

anio,filas,filas_con_region_final,filas_con_comuna_final,n_establecimientos
2020,6446646,6338147,6338147,607
2021,8816240,8747080,8747080,635
2022,8926307,8919840,8919840,638
2023,8899080,8899080,8899080,642
2024,8973229,8973229,8973229,640
2025,9142479,9142439,9142439,653


In [0]:
display(
    urg_valpo
    .groupBy("anio")
    .agg(
        F.countDistinct("idestablecimiento_norm").alias("n_establecimientos_valpo"),
        F.countDistinct("codigocomuna_final").alias("n_comunas_valpo"),
        F.sum("total_num").alias("total_atenciones")
    )
    .orderBy("anio")
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7529467704804100>, line 1
----> 1 display(
      2     urg_valpo
      3     .groupBy("anio")
      4     .agg(
      5         F.countDistinct("idestablecimiento_norm").alias("n_establecimientos_valpo"),
      6         F.countDistinct("codigocomuna_final").alias("n_comunas_valpo"),
      7         F.sum("total_num").alias("total_atenciones")
      8     )
      9     .orderBy("anio")
     10 )

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect

## 6. Visualizaciones

In [0]:
display(spark.table(BRONZE_SINCA_TABLE).limit(10))

fecha,estacion,contaminante,valor_nox,tipo_registro,source_file
2020-01-01,colmo,NOx,7.74861,no_validado,v_nox_diario_colmo_2018_2025.csv
2020-01-02,colmo,NOx,8.89061,no_validado,v_nox_diario_colmo_2018_2025.csv
2020-01-03,colmo,NOx,8.24716,no_validado,v_nox_diario_colmo_2018_2025.csv
2020-01-04,colmo,NOx,9.49925,no_validado,v_nox_diario_colmo_2018_2025.csv
2020-01-05,colmo,NOx,5.42806,no_validado,v_nox_diario_colmo_2018_2025.csv
2020-01-06,colmo,NOx,10.1788,no_validado,v_nox_diario_colmo_2018_2025.csv
2020-01-07,colmo,NOx,12.5873,no_validado,v_nox_diario_colmo_2018_2025.csv
2020-01-08,colmo,NOx,10.8252,no_validado,v_nox_diario_colmo_2018_2025.csv
2020-01-09,colmo,NOx,10.6378,no_validado,v_nox_diario_colmo_2018_2025.csv
2020-01-10,colmo,NOx,10.8555,no_validado,v_nox_diario_colmo_2018_2025.csv
